# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# # Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-4o-mini'
# openai = OpenAI()

In [2]:
# Constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [12]:
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'ht

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [21]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [6]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [7]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2024/12/21/

In [10]:
def get_links(url):
    website = Website(url)
    response = ollama_via_openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [8]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/agentica-org/DeepCoder-14B-Preview',
 '/meta-llama/Llama-4-Scout-17B-16E-Instruct',
 '/HiDream-ai/HiDream-I1-Full',
 '/moonshotai/Kimi-VL-A3B-Thinking',
 '/nvidia/Llama-3_1-Nemotron-Ultra-253B-v1',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/jamesliu1217/EasyControl_Ghibli',
 '/spaces/bytedance-research/UNO-FLUX',
 '/spaces/Efficient-Large-Model/SanaSprint',
 '/spaces/Stable-X/Hi3DGen',
 '/spaces',
 '/datasets/nvidia/OpenCodeReasoning',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset',
 '/datasets/open-thoughts/OpenThoughts2-1M',
 '/datasets/agentica-org/DeepCoder-Preview-Dataset',
 '/datasets/LLM360/MegaMath',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/a

In [13]:
get_links("https://huggingface.co")

{'links': [{'type': 'About page', 'url': 'https://huggingface.co/'},
  {'type': 'Careers/Jobs', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Pricing', 'url': 'https://ui.endpoints.huggingface.co/pricing'},
  {'type': 'Brand', 'url': 'https://huggingface.co/brand'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        try:
            result += f"\n\n{link['type']}\n"
            result += Website(link["url"]).get_contents()
        except Exception as e:
            print(f"\n\n error: {e}\n\n")
        
    return result

In [15]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'Company/About page', 'url': 'https://huggingface.co/'}, {'type': 'Models page', 'url': 'https://huggingface.co/models'}, {'type': 'Datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'Spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'Career/Jobs page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'Enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'Blog/Documentation page', 'url': 'https://docs.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


 error: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001AE5E0B5650

In [25]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'About page', 'url': 'https://huggingface.co'}, {'type': 'Company page', 'url': 'https://ui.endpoints.huggingface.co'}, {'type': 'Careers/Page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Documentation', 'url': 'https://docs.huggingface.co'}, {'type': 'Blog', 'url': 'https://blog.huggingface.co'}, {'type': 'Discussions', 'url': 'https://discuss.huggingface.co'}, {'type': 'Status', 'url': 'https://status.huggingface.co/'}, {'type': 'GitHub', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


 error: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001AE5D913DD0>: Failed to resolve 'docs.huggingface.co' ([Errno 11001] getaddrinfo failed)"))




 error: HTTPSConnectionPool(host='blo

'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nPosts\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nagentica-org/DeepCoder-14B-Preview\nUpdated\n4 days ago\n•\n6.87k\n•\n419\nmeta-llama/Llama-4-Scout-17B-16E-Instruct\nUpdated\n4 days ago\n•\n541k\n•\n755\nHiDream-ai/HiDream-I1-Full\nUpdated\nabout 10 hours ago\n•\n6.54k\n•\n331\nmoonshotai/Kimi-VL-A3B-Thinking\nUpdated\nabout 8 hours ago\n•\n4.69k\n•\n237\nnvidia/Llama-3_1-Nemotron-Ultra-253B-v1\nUpdated\n3 days ago\n•\n10.6k\n•\n211\nBrowse

In [19]:
def create_brochure(company_name, url):
    response = ollama_via_openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'About', 'url': '/'}, {'type': 'Company', 'url': 'https://brand.huggingface.co'}, {'type': 'Careers', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Products/Services', 'url': '/models'}, {'type': 'Datasets', 'url': '/datasets'}, {'type': 'Spaces', 'url': '/spaces'}, {'type': 'Knowledge Base/Academy', 'url': 'https://docs.huggingface.co'}, {'type': 'Blog', 'url': 'https://blog.huggingface.co'}, {'type': 'Discussions/Community', 'url': 'https://discuss.huggingface.co'}, {'type': 'Status/Support', 'url': 'https://status.huggingface.co'}]}


 error: Invalid URL '/': No scheme supplied. Perhaps you meant https:///?




 error: HTTPSConnectionPool(host='brand.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001AE5DAC8AD0>: Failed to resolve 'brand.huggingface.co' ([Errno 11001] getaddrinfo failed)"))




 error: Invalid URL '/models': No scheme suppli

**Hugging Face Brochure**
========================

**Welcome to the Future of AI**

Hugging Face is a leading platform for building and deploying artificial intelligence (AI) models. Our mission is to empower the global machine learning community by providing a collaborative environment, cutting-edge tools, and innovative solutions.

**Our Story**

Founded in 2016, Hugging Face has grown into one of the largest open-source AI frameworks, with over 50,000 organizations using our platform. Our team consists of passionate researchers, engineers, and enthusiasts dedicated to advancing the field of machine learning.

**Our Features**

* **Hugging Face Hub**: A community-driven repository hosting over 1 million pre-trained models, datasets, and applications.
* **Spaces**: A collaboration platform for hosts and users to work on unlimited public models, datasets, and applications.
* **Compute**: Optimized inference endpoints for deploying AI models with ease.
* **Enterprise**: Secure, scalable solutions for teams and organizations.

**Who We Are**

We are a diverse community of AI enthusiasts, researchers, engineers, and entrepreneurs. Our members include:

* Large corporations (Meta, Google, Amazon, Intel)
* Non-profit organizations
* Government agencies
* Small businesses and startups
* University researchers
* Hobbyists

**Stay Connected**

Follow us on social media to stay updated on the latest developments in AI and Hugging Face.

* Twitter: [@huggingface](https://twitter.com/huggingface)
* LinkedIn: [Hugging Face](https://www.linkedin.com/company/hugging-face/)
* GitHub: [HuggingFace](https://github.com/huggingface)

**Join Our Community**

Be part of the largest AI community in the world! Join our forums, contribute to open-source projects, and collaborate with others on innovative AI applications.

[Visit our Forum](https://forum.huggingface.co/)

**Career Opportunities**

Are you passionate about AI? Explore our current openings for engineers, researchers, and product managers. Apply today!

[See Our Openings](https://www.huggingface.co/careers)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = ollama_via_openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'company page', 'url': 'https://huggingface.co/'}, {'type': 'all company founders', 'url': 'https://discuss.huggingface.co/'}, {'type': 'brand', 'url': 'https://ui.endpoints.huggingface.co/chat'}, {'type': ' Careers/Jobs page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'About Us', 'url': 'https://discuss.huggingface.co/'}]}


# Hugging Face: Building the Future of AI

At its core, Hugging Face is a collaboration platform where machine learning (ML) enthusiasts come together to build, discover , and share models, datasets, and applications.

## Our Mission

Our mission is to accelerate the adoption of AI across industries and communities. We achieve this by providing a free and open-source platform for developers to create, share, and work with ML models.

## Key Features

*   **1 million+ Models**: Browse our vast model repository, and explore applications built on top of these pre-trained models.
*   **Collaboration Platform**: Connect with the global ML community, host and collaborate on public models, datasets, and applications.
*   **Optimized Inference Endpoints**: Deploy AI models to optimized inference endpoints with ease.
*   **Enterprise Solutions**: Get access to advanced features, support, and resources for businesses and organizations.

## Community

Hugging Face is a vibrant community of more than 50,000+ organizations, including well-known companies like Meta, Google, Amazon, Intel, Microsoft, and Grammarly, among others. Our members are passionate about advancing AI research and applications across various industries.

### Company Memberships

We're proud to be associated with some of the most influential players in the AI landscape:

*   **Company Members** *   [Meta](https://huggingface.co/meta) *   Google ([AI at Google](https://huggingface.co/google)) *   Amazon ([Amazon AI](https://huggingface.co.amazon)) *   Intel ([Intel AI](https://huggingface.co.intel)) *   Microsoft ([Microsoft Applied Science Faculty](https://huggingface.co/microsoft)) *   Grammarly (*[Grammarly Enterprise](https://huggingface.com/grammarly))

### Open Source

We're dedicated to building the foundation of ML tooling with our community.

*   **Transformers**: Our state-of-the-art library for PyTorch, TensorFlow, and JAX.
*   **Diffusers**: For fast and reliable Diffusion models in PyTorch.
*   **Safetensors**: A safe way to store/distribute neural network weights.

### Jobs and Careers

Ready to join our community? We have a range of job opportunities available for dedicated individuals passionate about AI and ML. Visit our [careers page](https://huggingface.com/careers) to explore the latest offerings.

## Stay Connected

Connect with us on social media:

*   **Twitter** @[HuggingFace](https://twitter.com/HuggingFace)
*   **GitHub** [Hugging Face Hub](https://github.com/huggingface/transformers)
*   **LinkedIn** [Hugging Face](https://www.linkedin.com/company/hugging-face/)
*   **Discord** [Hugging Face Community](https://discordapp.com/invite/huggingface)

[Download Hugging Face Hub](https://huggingface.co/models) and start building AI today!

Get the latest updates by visiting our blog: <https://blog.huggingface.co/>

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'home page', 'url': 'https://huggingface.co'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'docs hub', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'about the company founder(s)', 'url': 'https://nvidia.com/en-us/high-performance-computing/teams/llama-team.html'}, {'type': 'join/join process on Hugging Face Website', 'url': 'https://apply.workable.com/huggingface/'}]}


Hugging Face: Where AI Meets Hugs
==============================================

Welcome to the AI community where the future is built on hugging faces, not just emotional support. We're a platform where machine learning enthusiasts collaborate to build, discover, and unleash the power of artificial intelligence.

**Our Mission**
---------------

To accelerate AI innovation by providing a collaborative platform for researchers, developers, and innovators alike.

**About Us**
-------------

At Hugging Face, we believe that the future belongs to those who can hug their data with confidence. Our platform is home to over 1 million active models, datasets, and applications built on top of popular AI frameworks like PyTorch, TensorFlow, and JAX.

**Our People**
----------------

We're not just a company; we're a community. Our team consists of passionate AI researchers, engineers, and enthusiasts who share our vision of making AI accessible to everyone. From Meta, Amazon, Google, Intel, Microsoft, and more, you'll find over 50,000 organizations using Hugging Face.

**Our Open-Source Stack**
---------------------------

From Transformers to Diffusers, State-of-the-art ML for PyTorch, TensorFlow, JAX, we're building the foundation of AI tooling with our community. Explore our open-source projects:

*   Transformers (142,879)
*   Diffusers (28,550)
*   Safetensors (3,218)

[Download Our Open-Source Projects](/projects/)

**Join the Movement**
---------------------

If you're ready to join the Hugging Face squad and start building your AI portfolio, sign up today!

[Sign Up](/signup)

**Stay Connected**
-----------------

Follow us on social media to stay updated on our latest developments:

Twitter: <https://twitter.com/huggingface>
LinkedIn: <https://linkedin.com/company/hugging face>
Discord: <https://discord.gg/huggingface>

GitHub: <https://github.com/huggingface>

**Our Blogs**

*   [Hugging Face Blog](/community/blog)
*   [Learn How to Build Your AI Portfolio](https://www.huggingface.co/blog)

[Join Our Community Today!](#)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 2 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>